# Point Defects for Quantum Optics

This notebook covers the theory and practice of point defect calculations for solid-state quantum emitters. We focus on the NV centre in diamond as the primary example, with references to other systems.

**Topics covered:**
- Types of point defects
- Interacting periodic images
- Charge states and formation energy
- Defect electronic levels
- Magnetism and spin in defect systems
- The neutral vs charged defect approximation

---


## 1. Types of Point Defects

A **point defect** is a localised disruption to the crystal lattice involving one or a few atoms:

| Type | Description | Example |
|------|-------------|---------|
| **Vacancy** | Missing atom | V_C in diamond |
| **Substitutional** | Wrong atom on lattice site | N_C in diamond |
| **Interstitial** | Extra atom between lattice sites | H in Si |
| **Complex** | Combination | NV centre (N_C + V_C) |

For quantum optics we focus on **deep defects** — defects whose electronic levels sit well inside the bandgap, far from the band edges. These give sharp optical transitions and can act as single-photon emitters.


In [ ]:
from ase.build import bulk, make_supercell
from ase.geometry import get_distances
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt
import numpy as np

# Build the NV centre in diamond
prim = bulk('C', 'diamond', a=3.57)
sc = make_supercell(prim, np.diag([3, 3, 3]))

_, D = get_distances(sc.positions, sc.positions, cell=sc.cell, pbc=True)
np.fill_diagonal(D, np.inf)
vac_idx = int(np.argmin(D[0]))

sym = sc.get_chemical_symbols()
sym[0] = 'N'
sc.set_chemical_symbols(sym)
del sc[vac_idx]

print(f"NV centre supercell: {sc.get_chemical_formula()}")
print(f"Number of atoms: {len(sc)}")
print(f"Supercell dimensions: {sc.cell.lengths().round(2)} Å")

# Visualise
fig, ax = plt.subplots(figsize=(6, 6))
plot_atoms(sc, ax, rotation='10x,10y,0z', radii=0.4)
ax.set_title('NV centre in 3×3×3 diamond supercell\n(N in blue, vacancy at origin)')
ax.axis('off')
plt.tight_layout()
plt.show()


## 2. Interacting Periodic Images

With PBC, the defect interacts with its periodic images in neighbouring cells. This **finite-size error** affects:
- Total energy (spurious electrostatic interaction)
- Defect level position
- Elastic relaxation

The error decays with supercell size. For a **neutral defect** it falls off as 1/L^3. For a **charged defect** it falls off as 1/L — much slower, requiring either large supercells or an explicit finite-size correction (Freysoldt or Makov-Payne).

The minimum image distance is the key convergence parameter — it must be large enough that the defect does not "see" itself. Check this for your system before running expensive calculations.


In [ ]:
from ase.build import bulk, make_supercell
import numpy as np

prim = bulk('C', 'diamond', a=3.57)

print("Supercell size comparison for diamond NV centre:")
print(f"{'Size':10s} {'Atoms':8s} {'Min image dist':16s} {'Neutral OK?':12s} {'Charged OK?':12s}")
print("-" * 65)

for n in [2, 3, 4, 5]:
    sc = make_supercell(prim, np.diag([n, n, n]))
    min_dist = sc.cell[0, 0]
    neutral_ok  = "[yes]" if min_dist > 10 else "[no] "
    charged_ok  = "[yes]" if min_dist > 15 else "[no] "
    print(f"{n}x{n}x{n}:    {len(sc):4d}     {min_dist:8.2f} A         {neutral_ok}        {charged_ok}")

print()
print("Rule of thumb:")
print("  Neutral defects:  min image distance > 10 A")
print("  Charged defects:  min image distance > 15 A (or use charge correction)")
print()
print("Practical tip: always check this BEFORE running your calculation.")
print("A supercell that is too small will give wrong defect level positions.")
print()
print("Running GPAW in parallel (no script changes needed):")
print("  mpirun -n 4 gpaw python defect_dos.py")
print()
print("For a 3x3x3 diamond supercell (53 atoms):")
print("  1 core:  ~60-90 min")
print("  4 cores: ~20-30 min")
print("  8 cores: ~12-18 min")


## 3. Charge States and Formation Energy

The same defect can exist in multiple **charge states** — NV0 (neutral) and NV(-) (negative) for the NV centre, for example. Which charge state is stable depends on the **Fermi level** in the material.

The **defect formation energy** is:

$$E^f[X^q] = E_{\text{tot}}[X^q] - E_{\text{tot}}[\text{bulk}] - \sum_i n_i \mu_i + q(E_{\text{VBM}} + E_F) + E_{\text{corr}}$$

where:
- $q$ = charge state
- $n_i$ = number of atoms added/removed
- $\mu_i$ = chemical potential of species $i$
- $E_F$ = Fermi level (referenced to VBM)
- $E_{\text{corr}}$ = finite-size correction for charged supercells

The **thermodynamic transition level** $\varepsilon(q/q')$ is the Fermi level at which two charge states have equal formation energy.


In [ ]:
# Schematic formation energy diagram for NV centre
E_F = np.linspace(0, 5.47, 200)   # Fermi level from VBM to CBM (diamond gap)

# Schematic formation energies (slope = -q for charge state q)
# NV0:  q=0, flat line
# NV-:  q=-1, slope = +1 (energy decreases as E_F increases)
# NV2-: q=-2, slope = +2

E_NV0  = 4.0 + 0 * E_F        # neutral: flat
E_NVm  = 5.5 - 1 * E_F        # NV-: decreasing
E_NVmm = 8.5 - 2 * E_F        # NV2-: steeper decrease

# Transition levels (where lines cross)
# NV0/NV-: E_NV0 = E_NVm --> 4.0 = 5.5 - E_F --> E_F = 1.5 eV
# NV-/NV2-: E_NVm = E_NVmm --> 5.5 - E_F = 8.5 - 2E_F --> E_F = 3.0 eV

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(E_F, np.minimum(E_NV0, np.minimum(E_NVm, E_NVmm)),
        'k-', lw=3, label='Stable charge state')
ax.plot(E_F, E_NV0,  'b--', lw=1.5, alpha=0.6, label='NV0  (q=0)')
ax.plot(E_F, E_NVm,  'r--', lw=1.5, alpha=0.6, label='NV(-)  (q=-1)')
ax.plot(E_F, E_NVmm, 'g--', lw=1.5, alpha=0.6, label='NV2(-) (q=-2)')

ax.axvline(1.5, color='purple', ls=':', lw=1.5)
ax.axvline(3.0, color='purple', ls=':', lw=1.5)
ax.text(1.5, 0.5, 'ε(0/-)', ha='center', fontsize=10, color='purple')
ax.text(3.0, 0.5, 'ε(-/2-)', ha='center', fontsize=10, color='purple')

ax.fill_betweenx([0, 10], 0, 0, alpha=0)
ax.set_xlabel('Fermi level E_F (eV, relative to VBM)', fontsize=12)
ax.set_ylabel('Formation energy (eV)', fontsize=12)
ax.set_title('Schematic formation energy diagram — NV centre in diamond', fontsize=11)
ax.set_xlim(0, 5.47)
ax.set_ylim(0, 8)
ax.legend(fontsize=10)
ax.annotate('VBM', xy=(0, 0), xytext=(0.1, 7.5), fontsize=10)
ax.annotate('CBM', xy=(5.47, 0), xytext=(4.8, 7.5), fontsize=10)
plt.tight_layout()
plt.show()

print("NV(-) is stable when E_F > 1.5 eV above VBM")
print("This requires n-type doping or specific surface termination")


## 4. Defect Electronic Levels

The electronic signature of a defect is its **in-gap state** — a localised electronic level that sits inside the host bandgap. In the DOS, this appears as a sharp peak between the valence and conduction bands.

### Reading the spin-polarised DOS

When `spinpol=True`, GPAW computes separate spin-up (^) and spin-down (|) DOS. A **spin-asymmetric** in-gap peak is the hallmark of an open-shell defect:

- If spin-up and spin-down are **identical**: non-magnetic (S=0)
- If they differ: magnetic ground state (S≠0)
- The difference in peak heights/positions tells you about the spin state

### Defect level position

The defect level position relative to the VBM is approximately related to the optical transition energy (ZPL):

$$E_{\text{ZPL}} \approx E_{\text{defect level}} - \lambda$$

where $\lambda$ is the **reorganisation energy** — the energy the lattice gains by relaxing after the electronic transition. For systems with small electron-phonon coupling (small Huang-Rhys factor S), $\lambda$ is small and the defect level is a good approximation to the ZPL.


In [ ]:
# Schematic spin-polarised DOS for NV- in diamond
# Based on published GPAW/VASP calculations

E = np.linspace(-15, 10, 1000)

def gaussian(E, E0, sigma, weight=1.0):
    return weight * np.exp(-(E-E0)**2/(2*sigma**2))

# Valence band (both spins, symmetric)
vb = (gaussian(E, -10, 1.5, 20) + gaussian(E, -6, 1.5, 15) +
      gaussian(E, -3, 1.0, 10) + gaussian(E, -1, 0.5, 8))
vb *= (E < 0)

# Conduction band (both spins, symmetric)
cb = (gaussian(E, 5, 1.0, 10) + gaussian(E, 7, 1.0, 8)) * (E > 4.2)

# In-gap states (spin asymmetric for NV-)
# a1 states: both spins occupied, sit deeper in gap
a1_up = gaussian(E, -0.5, 0.08, 2.0)
a1_dn = gaussian(E, -0.5, 0.08, 2.0)

# e states: spin-up occupied, spin-down partially occupied
e_up = gaussian(E, 1.5, 0.08, 2.0)   # occupied
e_dn = gaussian(E, 2.5, 0.08, 1.0)   # unoccupied (above Fermi)

dos_up = vb + cb + a1_up + e_up
dos_dn = vb + cb + a1_dn + e_dn

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(E, dos_up,   alpha=0.4, color='steelblue')
ax.fill_between(E, -dos_dn,  alpha=0.4, color='tomato')
ax.plot(E, dos_up,  'steelblue', lw=1.5, label='Spin ^')
ax.plot(E, -dos_dn, 'tomato',    lw=1.5, label='Spin | (inverted)')
ax.axhline(0,   color='black', lw=0.8)
ax.axvline(0,   color='black', lw=1.2, ls='--', label='Fermi level / VBM')
ax.axvline(4.2, color='gray',  lw=1.0, ls=':',  label='CBM (~4.2 eV PBE)')
ax.axvline(1.945, color='green', lw=1.5, ls='--', label='Exp. ZPL (1.945 eV)')

ax.annotate('a₁ states\n(both spins)', xy=(−0.5, 2), xytext=(−3, 3),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9, ha='center')
ax.annotate('e^ occupied', xy=(1.5, 2), xytext=(0.5, 4),
            arrowprops=dict(arrowstyle='->', color='steelblue'),
            fontsize=9, color='steelblue')
ax.annotate('e| empty', xy=(2.5, -1), xytext=(3.5, -3),
            arrowprops=dict(arrowstyle='->', color='tomato'),
            fontsize=9, color='tomato')

ax.set_xlabel('Energy relative to VBM (eV)', fontsize=12)
ax.set_ylabel('DOS (arb. units)', fontsize=12)
ax.set_title('Schematic spin-polarised DOS — NV(-) in diamond', fontsize=12)
ax.set_xlim(-12, 8)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()


## 5. Magnetism and Spin in Defect Systems

### NV(-) electronic structure

NV(-) has 6 electrons in the defect complex (3 from C dangling bonds + 2 from N + 1 from charge). In C₃ᵥ symmetry these fill:

- **a₁** (low): 2 electrons, fully paired
- **a₁** (high): 2 electrons, fully paired  
- **e** (doubly degenerate): 2 electrons, by Hund's rule **parallel spins** --> S=1

The S=1 (spin triplet) ground state has three spin sublevels: m_s = 0, +1, -1. The zero-field splitting between m_s=0 and m_s=±1 is **2.87 GHz** — in the microwave range and measurable with standard electronics. This is what makes NV(-) a spin qubit.

### Counting electrons for defect charge state

| Species | Valence electrons | Contribution to defect |
|---------|------------------|----------------------|
| N (substitutional) | 5 | 2 (3 used in bonds to lattice) |
| 3× C neighbours | 4 each | 1 each (1 dangling bond) |
| Charge -1 | — | +1 |
| **Total NV(-)** | | **6** |


In [ ]:
# Setting up a spin-polarised defect calculation in GPAW
gpaw_setup = '''
Key GPAW settings for NV- in diamond:
=======================================

from gpaw import GPAW, Mixer

# Set initial magnetic moments BEFORE attaching calculator
magmoms = [0.0] * len(atoms)
magmoms[0] = 2.0   # N atom: total moment = 2 (S=1)
atoms.set_initial_magnetic_moments(magmoms)

# GPAW calculator
calc = GPAW(
    mode="lcao", basis="dzp", xc="PBE",
    kpts={"size": (1,1,1), "gamma": True},
    spinpol=True,       # separate spin channels
    symmetry="off",     # defect breaks crystal symmetry
    charge=-1,          # NV- charge state
    mixer=Mixer(beta=0.02, nmaxold=8, weight=100),
    convergence={"energy": 0.005, "density": 0.005, "eigenstates": 1e-6},
    txt="-",
)

# Check converged magnetic moment in output:
# iter:  50  ...  +2.001  <-- correct (S=1)
# iter:  50  ...  +0.001  <-- wrong (S=0), restart with different initial moments
'''
print(gpaw_setup)


## 6. The Neutral Defect Approximation and Its Limits

In this course we primarily calculate **neutral defects** for simplicity. This is an important approximation to understand.

### What neutral calculations miss

1. **Wrong charge state**: NV0 has different electronic structure from NV(-). The in-gap levels, spin state, and optical transition are all different.

2. **No charge correction needed**: charged supercells require Freysoldt or Makov-Payne correction to remove spurious image interactions. Neutral supercells do not.

3. **Formation energy**: without the charge term $q(E_{\text{VBM}} + E_F)$ we can't compute which charge state is stable.

### When neutral is acceptable

- For **geometry**: NV0 and NV(-) geometries are similar (within ~1% bond lengths)
- For **qualitative DOS**: an in-gap state exists in both charge states
- For **strain tuning**: the strain response is qualitatively similar

### Questions to consider

Use this framework to assess your results:

1. What charge state is relevant for quantum optics applications of your system?
2. How does the neutral approximation affect the defect level position you calculated?
3. Would the in-gap state move up or down in energy in the correct charge state?


In [ ]:
# Summary: charged vs neutral defect calculations

print("=" * 60)
print("Neutral vs Charged Defect Calculations")
print("=" * 60)
print()
print("NEUTRAL (what we calculate in this course):")
print("  [yes] No charge correction needed")
print("  [yes] Faster convergence")  
print("  [yes] Good geometry approximation")
print("  [yes] Qualitative in-gap state visible")
print("  [no] May be wrong charge state for quantum optics")
print("  [no] Cannot compute formation energy vs Fermi level")
print("  [no] Spin state may differ from relevant charge state")
print()
print("CHARGED (research-level calculation):")
print("  [yes] Correct charge state for application")
print("  [yes] Accurate defect level position")
print("  [yes] Can compute thermodynamic transition levels")
print("  [no] Requires Freysoldt/Makov-Payne charge correction")
print("  [no] Larger supercell needed (charged image interaction ~1/L)")
print("  [no] More complex convergence")
print()
print("For this course: use neutral, discuss limitations in Part 3.4")
print("For research: use charged with corrections (e.g. shakenbreak + py-sc-fermi)")
